# Rakuten - Classification Multimodale E-commerce
## Phase 1 - Preprocessing des données textuelles

In [1]:
# ============================================================
# CELLULE 1 — Imports
# ============================================================
import pandas as pd
import nltk
import spacy
from bs4 import BeautifulSoup
from nltk.corpus import stopwords

nltk.download('stopwords')

stop_words = set(stopwords.words('french'))
nlp = spacy.load('fr_core_news_sm')

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/ilansarfati/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
# ============================================================
# CELLULE 2 — Chargement des données
# ============================================================
df_X = pd.read_csv("/Users/ilansarfati/Desktop/RAKUTEN/X_train_update.csv", index_col=0)
df_Y = pd.read_csv("/Users/ilansarfati/Desktop/RAKUTEN/Y_train_CVw08PX.csv", index_col=0)

print(f"Taille du dataset : {df_X.shape}")
df_X[["designation", "description"]].head(5)

Taille du dataset : (84916, 4)


,designation,description
0,Olivia: Personalisiertes Notizbuch / 150 Seite...,NaN
1,Journal Des Arts (Le) N° 133 Du 28/09/2001 - L...,NaN
2,Grand Stylet Ergonomique Bleu Gamepad Nintendo...,PILOT STYLE Touch Pen de marque Speedlink est ...
3,Peluche Donald - Europe - Disneyland 2000 (Mar...,NaN
4,La Guerre Des Tuques,Luc a des id&eacute;es de grandeur. Il veut or...


In [3]:
# ============================================================
# CELLULE 3 — Exploration des données
# ============================================================

# Distribution des classes
print("Distribution des classes :")
print(df_Y["prdtypecode"].value_counts())
print(f"\nNombre de classes uniques : {df_Y['prdtypecode'].nunique()}")

# Valeurs manquantes
print(f"\nNombre de descriptions manquantes : {df_X['description'].isna().sum()}")
print(f"Pourcentage : {round(df_X['description'].isna().sum() / len(df_X) * 100, 1)}%")

Distribution des classes :
prdtypecode
2583    10209
1560     5073
1300     5045
2060     4993
2522     4989
1280     4870
2403     4774
2280     4760
1920     4303
1160     3953
1320     3241
10       3116
2705     2761
1140     2671
2582     2589
40       2508
2585     2496
1302     2491
1281     2070
50       1681
2462     1421
2905      872
60        832
2220      824
1301      807
1940      803
1180      764
Name: count, dtype: int64

Nombre de classes uniques : 27

Nombre de descriptions manquantes : 29800
Pourcentage : 35.1%


In [4]:
# ============================================================
# CELLULE 4 — Fonctions de preprocessing
# ============================================================

def clean_html(text):
    """Supprime les balises HTML et décode les entités HTML"""
    soup = BeautifulSoup(text, "html.parser")
    return soup.get_text()

def remove_stopwords(text):
    """Supprime les mots vides (stopwords) français"""
    mots = text.split()
    mots_filtre = [mot for mot in mots if mot not in stop_words]
    return " ".join(mots_filtre)

def lemmatize(text):
    """Ramène chaque mot à sa forme racine (lemme)"""
    doc = nlp(text)
    lemmes = [token.lemma_ for token in doc]
    return " ".join(lemmes)

In [ ]:
# ============================================================
# CELLULE 5 — Pipeline de preprocessing
# ============================================================

# Étape 1 : Fusionner designation + description
df_X["text"] = df_X["designation"] + " " + df_X["description"].fillna("")
print("✅ Étape 1 : Fusion designation + description")

# Étape 2 : Nettoyer le HTML
df_X["text"] = df_X["text"].apply(clean_html)
print("✅ Étape 2 : Nettoyage HTML")

# Étape 3 : Lowercasing
df_X["text"] = df_X["text"].str.lower()
print("✅ Étape 3 : Lowercasing")

# Étape 4 : Suppression des stopwords
df_X["text"] = df_X["text"].apply(remove_stopwords)
print("✅ Étape 4 : Suppression des stopwords")

# Étape 5 : Lemmatisation
df_X["text"] = df_X["text"].apply(lemmatize)
print("✅ Étape 5 : Lemmatisation")

# Aperçu du résultat
df_X["text"].head(10)

✅ Étape 1 : Fusion designation + description


/var/folders/8g/nlbz_rzd6lz9xdj64s6mwp2m0000gn/T/ipykernel_1626/3165547566.py:7: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  soup = BeautifulSoup(text, "html.parser")


✅ Étape 2 : Nettoyage HTML
✅ Étape 3 : Lowercasing
✅ Étape 4 : Suppression des stopwords


In [ ]:
# ============================================================
# CELLULE 6 — Sauvegarde
# ============================================================
df_X.to_csv("df_X_clean.csv")
print("✅ Fichier sauvegardé : df_X_clean.csv")